# UMTAM vs Fisher-Weighted Merging: Comparative Study
## 🚀 Optimized for Google Colab T4 GPU (16GB VRAM)

### Research Question
**Reviewer Comment:** "It remains unclear whether UMTAM maintains a significant performance advantage if traditional Fisher-weighted methods are provided with equally precise Fisher information."

### Optimizations for T4
- ✅ Mixed precision (FP16) training
- ✅ Reduced batch size (16) with gradient accumulation
- ✅ Batched Fisher computation (not per-sample)
- ✅ Aggressive memory management
- ✅ Reduced sample sizes for tractability

### Expected Runtime: ~2-3 hours on T4

---
## 📦 Setup

In [ ]:
# Install packages
!pip install -q transformers datasets torch evaluate matplotlib seaborn scipy tqdm accelerate
print("✓ Packages installed")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from datasets import load_dataset
import evaluate
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import json
import copy
import warnings
import gc
import time
from collections import defaultdict
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✓ Device: {device}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ Memory: {gpu_mem:.1f} GB")
    
    # Warn if not T4
    if gpu_mem < 14:
        print("⚠️ WARNING: Less than 14GB VRAM detected. May need to reduce batch size further.")
    elif gpu_mem > 20:
        print("ℹ️ INFO: More than 20GB VRAM detected. You could increase batch sizes for faster training.")

In [ ]:
def clear_memory():
    """Aggressive memory clearing for T4"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def print_gpu_memory(label=""):
    """Print current GPU memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"  GPU Memory {label}: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

print("✓ Memory utilities defined")

---
## ⚙️ Configuration (T4 Optimized)

In [ ]:
CONFIG = {
    # Model
    'model_name': 'bert-base-uncased',
    
    # Tasks
    'tasks': {
        'rte': {
            'num_labels': 2,
            'metric': 'accuracy',
            'glue_name': 'rte',
            'text_fields': ('sentence1', 'sentence2'),
        },
        'mrpc': {
            'num_labels': 2,
            'metric': 'f1',
            'glue_name': 'mrpc',
            'text_fields': ('sentence1', 'sentence2'),
        },
        'qnli': {
            'num_labels': 2,
            'metric': 'accuracy',
            'glue_name': 'qnli',
            'text_fields': ('question', 'sentence'),
        },
        'cola': {
            'num_labels': 2,
            'metric': 'matthews_correlation',
            'glue_name': 'cola',
            'text_fields': ('sentence', None),
        },
    },
    
    # Training - T4 OPTIMIZED
    'max_length': 128,
    'batch_size': 16,              # Reduced from 32
    'gradient_accumulation': 2,    # Effective batch = 32
    'num_epochs': 3,
    'lr': 2e-5,
    'use_amp': True,               # Mixed precision
    
    # Merging
    'sparsity_k': 20,
    
    # Fisher computation - T4 OPTIMIZED
    'fisher_samples': [100, 500],  # Removed 'Full' for speed
    'fisher_batch_size': 8,        # Smaller batch for Fisher
    'fisher_train_samples': 1000,  # Reduced from 2000
    
    # Data limits - T4 OPTIMIZED
    'max_train_samples': 3000,     # Reduced from 5000
    'max_eval_samples': 500,       # Reduced from 1000
    
    # Other
    'seed': 42,
}

# Set seed
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

print("✓ T4-Optimized Configuration:")
print(f"  Model: {CONFIG['model_name']}")
print(f"  Tasks: {list(CONFIG['tasks'].keys())}")
print(f"  Batch size: {CONFIG['batch_size']} (effective: {CONFIG['batch_size'] * CONFIG['gradient_accumulation']})")
print(f"  Mixed precision: {CONFIG['use_amp']}")
print(f"  Max train samples: {CONFIG['max_train_samples']}")
print(f"  Fisher samples: {CONFIG['fisher_samples']} + Train({CONFIG['fisher_train_samples']})")

---
## 📚 Load Datasets

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

def load_and_prepare_dataset(task_name, task_config):
    """Load and tokenize a GLUE dataset"""
    print(f"\nLoading {task_name.upper()}...")
    dataset = load_dataset('glue', task_config['glue_name'])
    
    text1, text2 = task_config['text_fields']
    
    def tokenize_fn(examples):
        if text2 is None:
            result = tokenizer(
                examples[text1],
                padding='max_length',
                truncation=True,
                max_length=CONFIG['max_length']
            )
        else:
            result = tokenizer(
                examples[text1],
                examples[text2],
                padding='max_length',
                truncation=True,
                max_length=CONFIG['max_length']
            )
        result['labels'] = examples['label']
        return result
    
    tokenized = dataset.map(tokenize_fn, batched=True)
    
    # Subsample
    train_data = tokenized['train'].shuffle(seed=CONFIG['seed'])
    if CONFIG['max_train_samples'] and len(train_data) > CONFIG['max_train_samples']:
        train_data = train_data.select(range(CONFIG['max_train_samples']))
    
    eval_data = tokenized['validation']
    if CONFIG['max_eval_samples'] and len(eval_data) > CONFIG['max_eval_samples']:
        eval_data = eval_data.select(range(CONFIG['max_eval_samples']))
    
    train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    eval_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    print(f"  Train: {len(train_data)}, Eval: {len(eval_data)}")
    return train_data, eval_data

# Load all datasets
datasets = {}
for task_name, task_config in CONFIG['tasks'].items():
    train_data, eval_data = load_and_prepare_dataset(task_name, task_config)
    datasets[task_name] = {'train': train_data, 'eval': eval_data}

print(f"\n✓ Loaded {len(datasets)} datasets")
clear_memory()

---
## 🏋️ Training with Mixed Precision + UMTAM Saliency

In [ ]:
def train_task_expert(task_name, task_config, train_dataset, base_state_dict):
    """
    Train a task-specific expert with T4 optimizations:
    - Mixed precision (FP16)
    - Gradient accumulation
    - Aggressive memory management
    """
    print(f"\n{'='*60}")
    print(f"Training {task_name.upper()} Expert")
    print(f"{'='*60}")
    print_gpu_memory("before loading model")
    
    # Create model
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=task_config['num_labels']
    )
    
    # Load shared initialization
    model_dict = model.state_dict()
    for name, param in base_state_dict.items():
        if name in model_dict and 'classifier' not in name:
            model_dict[name] = param.clone()
    model.load_state_dict(model_dict)
    
    model.to(device)
    model.train()
    print_gpu_memory("after loading model")
    
    optimizer = AdamW(model.parameters(), lr=CONFIG['lr'])
    scaler = GradScaler() if CONFIG['use_amp'] else None
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        pin_memory=True,
        num_workers=2
    )
    
    # Store w0
    w0_dict = {}
    for name, param in model.named_parameters():
        if len(param.shape) == 2 and 'classifier' not in name:
            w0_dict[name] = param.data.clone().cpu()
    
    # Initialize curvature tracking
    curvature_dict = {}
    for name, param in model.named_parameters():
        if name in w0_dict:
            m, n = param.shape
            curvature_dict[name] = {
                'R': torch.zeros(m, device=device, dtype=torch.float32),
                'C': torch.zeros(n, device=device, dtype=torch.float32),
            }
    
    grad_accum = CONFIG['gradient_accumulation']
    
    # Training loop
    for epoch in range(CONFIG['num_epochs']):
        total_loss = 0
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']}")
        
        for step, batch in enumerate(pbar):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            
            # Mixed precision forward
            if CONFIG['use_amp']:
                with autocast():
                    outputs = model(**batch)
                    loss = outputs.loss / grad_accum
                scaler.scale(loss).backward()
            else:
                outputs = model(**batch)
                loss = outputs.loss / grad_accum
                loss.backward()
            
            # Accumulate curvature (in FP32 for stability)
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in curvature_dict and param.grad is not None:
                        grad_sq = param.grad.float() ** 2
                        curvature_dict[name]['R'] += grad_sq.sum(dim=1)
                        curvature_dict[name]['C'] += grad_sq.sum(dim=0)
            
            # Gradient accumulation step
            if (step + 1) % grad_accum == 0:
                if CONFIG['use_amp']:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad()
            
            total_loss += loss.item() * grad_accum
            pbar.set_postfix({'loss': f'{loss.item() * grad_accum:.4f}'})
        
        print(f"  Epoch {epoch+1} avg loss: {total_loss / len(train_loader):.4f}")
        print_gpu_memory(f"end of epoch {epoch+1}")
    
    # Extract Adam's v_T
    adam_v = {}
    param_to_name = {id(p): n for n, p in model.named_parameters()}
    for param_group in optimizer.param_groups:
        for p in param_group['params']:
            if id(p) in param_to_name:
                name = param_to_name[id(p)]
                if name in w0_dict:
                    state = optimizer.state[p]
                    if 'exp_avg_sq' in state:
                        adam_v[name] = state['exp_avg_sq'].cpu().clone()
    
    # Compute task vector
    task_vector = {}
    for name in w0_dict.keys():
        param = dict(model.named_parameters())[name]
        task_vector[name] = (param.data.cpu().float() - w0_dict[name]).clone()
    
    # Compute UMTAM saliency
    saliency_dict = {}
    for name in w0_dict.keys():
        param = dict(model.named_parameters())[name]
        w0 = w0_dict[name].to(device)
        
        delta_sq = (param.data.float() - w0) ** 2
        R = curvature_dict[name]['R'].clamp(min=1e-8)
        C = curvature_dict[name]['C'].clamp(min=1e-8)
        curvature = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
        saliency_dict[name] = (delta_sq * curvature).cpu()
    
    # Move curvature to CPU
    curvature_cpu = {}
    for name in curvature_dict:
        curvature_cpu[name] = {
            'R': curvature_dict[name]['R'].cpu(),
            'C': curvature_dict[name]['C'].cpu(),
        }
    
    trained_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    # Cleanup
    del model, optimizer, scaler
    clear_memory()
    
    print(f"✓ {task_name.upper()} expert trained")
    print_gpu_memory("after cleanup")
    
    return {
        'state_dict': trained_state,
        'task_vector': task_vector,
        'saliency': saliency_dict,
        'curvature': curvature_cpu,
        'adam_v': adam_v,
        'w0': w0_dict,
    }

print("✓ Training function defined (T4 optimized)")

---
## 🔬 Fisher Information (Batched for T4)

In [ ]:
def compute_fisher_information_batched(model, dataloader, num_samples=None, layer_names=None):
    """
    Compute empirical Fisher information using BATCHED computation.
    Much faster than per-sample on T4.
    
    Uses the approximation: Fisher ≈ E[grad²] computed over batches
    This is slightly less accurate than per-sample but much faster.
    """
    model.eval()
    
    fisher_dict = {}
    for name, param in model.named_parameters():
        if layer_names is None or name in layer_names:
            if len(param.shape) == 2:
                fisher_dict[name] = torch.zeros_like(param, dtype=torch.float32)
    
    sample_count = 0
    total_samples = num_samples if num_samples else len(dataloader.dataset)
    
    pbar = tqdm(dataloader, desc=f"Computing Fisher ({total_samples} samples)")
    
    for batch in pbar:
        if num_samples and sample_count >= num_samples:
            break
        
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        batch_size = batch['input_ids'].shape[0]
        
        model.zero_grad()
        
        # Forward with mixed precision
        if CONFIG['use_amp']:
            with autocast():
                outputs = model(**batch)
                # Use cross-entropy loss (equivalent to -log p(y|x))
                loss = outputs.loss
        else:
            outputs = model(**batch)
            loss = outputs.loss
        
        loss.backward()
        
        # Accumulate squared gradients (normalized by batch size)
        with torch.no_grad():
            for name, param in model.named_parameters():
                if name in fisher_dict and param.grad is not None:
                    # Scale by batch_size since loss is averaged
                    fisher_dict[name] += (param.grad.float() ** 2) * batch_size
        
        sample_count += batch_size
        pbar.set_postfix({'samples': min(sample_count, total_samples)})
        
        if num_samples and sample_count >= num_samples:
            break
    
    # Normalize
    for name in fisher_dict:
        fisher_dict[name] = fisher_dict[name] / sample_count
    
    model.train()
    return fisher_dict


def compute_fisher_for_expert(task_name, task_config, expert, dataset, num_samples=None):
    """
    Compute Fisher information for a trained expert.
    """
    label = num_samples if num_samples else 'full'
    print(f"\nComputing Fisher for {task_name.upper()} ({label} samples)...")
    start_time = time.time()
    print_gpu_memory("before Fisher")
    
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=task_config['num_labels']
    )
    model.load_state_dict(expert['state_dict'])
    model.to(device)
    
    dataloader = DataLoader(
        dataset, 
        batch_size=CONFIG['fisher_batch_size'],
        shuffle=False,
        pin_memory=True
    )
    
    layer_names = list(expert['task_vector'].keys())
    fisher = compute_fisher_information_batched(model, dataloader, num_samples, layer_names)
    fisher_cpu = {name: f.cpu() for name, f in fisher.items()}
    
    elapsed = time.time() - start_time
    print(f"  ✓ Fisher computed in {elapsed:.1f}s")
    
    del model
    clear_memory()
    
    return fisher_cpu

print("✓ Fisher computation functions defined (batched for T4)")

---
## 🔀 Merging Methods

In [ ]:
def merge_ties(base_state, task_experts, k_percent=20):
    """TIES-Merging: Magnitude-based baseline"""
    merged_state = copy.deepcopy(base_state)
    layer_names = list(task_experts[list(task_experts.keys())[0]]['task_vector'].keys())
    task_names = list(task_experts.keys())
    
    for name in layer_names:
        masks = {}
        trimmed_vectors = {}
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            magnitude = tv.abs().flatten()
            k = int(len(magnitude) * k_percent / 100)
            threshold = torch.kthvalue(magnitude, len(magnitude) - k).values
            mask = (tv.abs() >= threshold).float()
            masks[task_name] = mask
            trimmed_vectors[task_name] = tv * mask
        
        pos_mass = torch.zeros_like(trimmed_vectors[task_names[0]])
        neg_mass = torch.zeros_like(trimmed_vectors[task_names[0]])
        
        for task_name in task_names:
            tv = trimmed_vectors[task_name]
            pos_mass += (tv > 0).float() * tv.abs()
            neg_mass += (tv < 0).float() * tv.abs()
        
        elected_sign = torch.sign(pos_mass - neg_mass)
        
        merged_delta = torch.zeros_like(trimmed_vectors[task_names[0]])
        count = torch.zeros_like(trimmed_vectors[task_names[0]])
        
        for task_name in task_names:
            tv = trimmed_vectors[task_name]
            agree_mask = (torch.sign(tv) == elected_sign).float() * masks[task_name]
            merged_delta += tv * agree_mask
            count += agree_mask
        
        merged_delta = torch.where(count > 0, merged_delta / count.clamp(min=1), merged_delta)
        
        if name in merged_state:
            merged_state[name] = merged_state[name] + merged_delta
    
    return merged_state


def merge_umtam(base_state, task_experts, k_percent=20):
    """UMTAM: Trajectory-informed curvature-aware merging"""
    merged_state = copy.deepcopy(base_state)
    layer_names = list(task_experts[list(task_experts.keys())[0]]['task_vector'].keys())
    task_names = list(task_experts.keys())
    
    for name in layer_names:
        masks = {}
        
        for task_name in task_names:
            saliency = task_experts[task_name]['saliency'][name]
            flat_saliency = saliency.flatten()
            k = int(len(flat_saliency) * k_percent / 100)
            threshold = torch.kthvalue(flat_saliency, len(flat_saliency) - k).values
            masks[task_name] = (saliency >= threshold).float()
        
        pos_mass = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        neg_mass = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            saliency = task_experts[task_name]['saliency'][name]
            mask = masks[task_name]
            pos_mass += (tv > 0).float() * saliency * mask
            neg_mass += (tv < 0).float() * saliency * mask
        
        elected_sign = torch.sign(pos_mass - neg_mass)
        
        weighted_sum = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        weight_total = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            mask = masks[task_name]
            agree_mask = (torch.sign(tv) == elected_sign).float() * mask
            
            R = task_experts[task_name]['curvature'][name]['R'].clamp(min=1e-8)
            C = task_experts[task_name]['curvature'][name]['C'].clamp(min=1e-8)
            curvature_weight = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
            curvature_weight = curvature_weight / (curvature_weight.max() + 1e-8)
            
            weighted_sum += tv * agree_mask * curvature_weight
            weight_total += agree_mask * curvature_weight
        
        merged_delta = torch.where(
            weight_total > 0,
            weighted_sum / weight_total.clamp(min=1e-8),
            torch.zeros_like(weighted_sum)
        )
        
        if name in merged_state:
            merged_state[name] = merged_state[name] + merged_delta
    
    return merged_state


def merge_fisher_weighted(base_state, task_experts, fisher_info, k_percent=20):
    """Fisher-weighted merging with post-hoc Fisher information"""
    merged_state = copy.deepcopy(base_state)
    layer_names = list(task_experts[list(task_experts.keys())[0]]['task_vector'].keys())
    task_names = list(task_experts.keys())
    
    for name in layer_names:
        masks = {}
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            fisher = fisher_info[task_name].get(name, torch.ones_like(tv))
            saliency = (tv ** 2) * fisher.clamp(min=1e-8)
            
            flat_saliency = saliency.flatten()
            k = int(len(flat_saliency) * k_percent / 100)
            threshold = torch.kthvalue(flat_saliency, len(flat_saliency) - k).values
            masks[task_name] = (saliency >= threshold).float()
        
        pos_mass = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        neg_mass = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            fisher = fisher_info[task_name].get(name, torch.ones_like(tv))
            saliency = (tv ** 2) * fisher.clamp(min=1e-8)
            mask = masks[task_name]
            pos_mass += (tv > 0).float() * saliency * mask
            neg_mass += (tv < 0).float() * saliency * mask
        
        elected_sign = torch.sign(pos_mass - neg_mass)
        
        weighted_sum = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        weight_total = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            fisher = fisher_info[task_name].get(name, torch.ones_like(tv))
            mask = masks[task_name]
            agree_mask = (torch.sign(tv) == elected_sign).float() * mask
            
            fisher_weight = fisher.clamp(min=1e-8)
            fisher_weight = fisher_weight / (fisher_weight.max() + 1e-8)
            
            weighted_sum += tv * agree_mask * fisher_weight
            weight_total += agree_mask * fisher_weight
        
        merged_delta = torch.where(
            weight_total > 0,
            weighted_sum / weight_total.clamp(min=1e-8),
            torch.zeros_like(weighted_sum)
        )
        
        if name in merged_state:
            merged_state[name] = merged_state[name] + merged_delta
    
    return merged_state


def merge_ota_style(base_state, task_experts, k_percent=20):
    """OTA-style: Uses Adam's v_T (static curvature at training end)"""
    merged_state = copy.deepcopy(base_state)
    layer_names = list(task_experts[list(task_experts.keys())[0]]['task_vector'].keys())
    task_names = list(task_experts.keys())
    
    for name in layer_names:
        masks = {}
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            adam_v = task_experts[task_name]['adam_v'].get(name, torch.ones_like(tv))
            saliency = (tv ** 2) * adam_v.clamp(min=1e-8)
            
            flat_saliency = saliency.flatten()
            k = int(len(flat_saliency) * k_percent / 100)
            threshold = torch.kthvalue(flat_saliency, len(flat_saliency) - k).values
            masks[task_name] = (saliency >= threshold).float()
        
        pos_mass = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        neg_mass = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            adam_v = task_experts[task_name]['adam_v'].get(name, torch.ones_like(tv))
            saliency = (tv ** 2) * adam_v.clamp(min=1e-8)
            mask = masks[task_name]
            pos_mass += (tv > 0).float() * saliency * mask
            neg_mass += (tv < 0).float() * saliency * mask
        
        elected_sign = torch.sign(pos_mass - neg_mass)
        
        weighted_sum = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        weight_total = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            adam_v = task_experts[task_name]['adam_v'].get(name, torch.ones_like(tv))
            mask = masks[task_name]
            agree_mask = (torch.sign(tv) == elected_sign).float() * mask
            
            v_weight = adam_v.clamp(min=1e-8)
            v_weight = v_weight / (v_weight.max() + 1e-8)
            
            weighted_sum += tv * agree_mask * v_weight
            weight_total += agree_mask * v_weight
        
        merged_delta = torch.where(
            weight_total > 0,
            weighted_sum / weight_total.clamp(min=1e-8),
            torch.zeros_like(weighted_sum)
        )
        
        if name in merged_state:
            merged_state[name] = merged_state[name] + merged_delta
    
    return merged_state

print("✓ Merging functions defined")

---
## 📊 Evaluation

In [ ]:
def evaluate_merged_model(merged_state, base_state_dict, task_name, task_config, eval_dataset):
    """Evaluate merged model on a task"""
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=task_config['num_labels']
    )
    model.load_state_dict(merged_state, strict=False)
    model.to(device)
    model.eval()
    
    eval_loader = DataLoader(eval_dataset, batch_size=CONFIG['batch_size'])
    metric = evaluate.load('glue', task_config['glue_name'])
    
    with torch.no_grad():
        for batch in eval_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            if CONFIG['use_amp']:
                with autocast():
                    outputs = model(**batch)
            else:
                outputs = model(**batch)
            predictions = outputs.logits.argmax(dim=-1)
            metric.add_batch(predictions=predictions, references=batch['labels'])
    
    results = metric.compute()
    score = results.get(task_config['metric'], results.get('accuracy', 0))
    
    del model
    clear_memory()
    
    return score

print("✓ Evaluation function defined")

---
## 🚀 Run Experiment

In [ ]:
# Load base model
print("Loading base model...")
base_model = AutoModelForSequenceClassification.from_pretrained(CONFIG['model_name'], num_labels=2)
base_state_dict = {k: v.cpu().clone() for k, v in base_model.state_dict().items()}
del base_model
clear_memory()
print(f"✓ Base model loaded")

In [ ]:
# Train task experts
task_experts = {}
training_times = {}

for task_name, task_config in CONFIG['tasks'].items():
    start_time = time.time()
    expert = train_task_expert(
        task_name, task_config,
        datasets[task_name]['train'],
        base_state_dict
    )
    task_experts[task_name] = expert
    training_times[task_name] = time.time() - start_time
    clear_memory()

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")
for task_name, t in training_times.items():
    print(f"  {task_name.upper()}: {t/60:.1f} min")
print(f"  Total: {sum(training_times.values())/60:.1f} min")

In [ ]:
# Evaluate individual experts
print("\nEvaluating individual experts (upper bound)...")
individual_results = {}

for task_name, task_config in CONFIG['tasks'].items():
    score = evaluate_merged_model(
        task_experts[task_name]['state_dict'],
        base_state_dict,
        task_name,
        task_config,
        datasets[task_name]['eval']
    )
    individual_results[task_name] = score
    print(f"  {task_name.upper()}: {score:.4f}")

print(f"\n  Average: {np.mean(list(individual_results.values())):.4f}")

In [ ]:
# Compute Fisher information
fisher_info = {}
fisher_times = {}

# Fisher on validation set at different sample sizes
for num_samples in CONFIG['fisher_samples']:
    label = f"Fisher-{num_samples}"
    print(f"\n{'='*60}")
    print(f"Computing {label}")
    print(f"{'='*60}")
    
    fisher_info[label] = {}
    start_time = time.time()
    
    for task_name, task_config in CONFIG['tasks'].items():
        fisher = compute_fisher_for_expert(
            task_name, task_config,
            task_experts[task_name],
            datasets[task_name]['eval'],
            num_samples=num_samples
        )
        fisher_info[label][task_name] = fisher
        clear_memory()
    
    fisher_times[label] = time.time() - start_time
    print(f"  Total: {fisher_times[label]/60:.1f} min")

# Fisher on training set
print(f"\n{'='*60}")
print(f"Computing Fisher-Train ({CONFIG['fisher_train_samples']} samples)")
print(f"{'='*60}")

fisher_info['Fisher-Train'] = {}
start_time = time.time()

for task_name, task_config in CONFIG['tasks'].items():
    fisher = compute_fisher_for_expert(
        task_name, task_config,
        task_experts[task_name],
        datasets[task_name]['train'],
        num_samples=CONFIG['fisher_train_samples']
    )
    fisher_info['Fisher-Train'][task_name] = fisher
    clear_memory()

fisher_times['Fisher-Train'] = time.time() - start_time
print(f"  Total: {fisher_times['Fisher-Train']/60:.1f} min")

In [ ]:
# Run all merging methods
all_results = {}
merge_times = {}
k_percent = CONFIG['sparsity_k']

methods_to_run = [
    ('TIES', lambda: merge_ties(base_state_dict, task_experts, k_percent)),
    ('UMTAM', lambda: merge_umtam(base_state_dict, task_experts, k_percent)),
    ('OTA-style', lambda: merge_ota_style(base_state_dict, task_experts, k_percent)),
]

# Add Fisher methods
for fisher_label in fisher_info.keys():
    methods_to_run.append(
        (fisher_label, lambda fl=fisher_label: merge_fisher_weighted(base_state_dict, task_experts, fisher_info[fl], k_percent))
    )

print(f"\n{'='*70}")
print(f"MERGING AND EVALUATION (k={k_percent}%)")
print(f"{'='*70}")

for i, (method_name, merge_fn) in enumerate(methods_to_run):
    print(f"\n[{i+1}/{len(methods_to_run)}] {method_name}...")
    
    start = time.time()
    merged = merge_fn()
    merge_times[method_name] = time.time() - start
    
    all_results[method_name] = {}
    for task_name, task_config in CONFIG['tasks'].items():
        score = evaluate_merged_model(merged, base_state_dict, task_name, task_config, datasets[task_name]['eval'])
        all_results[method_name][task_name] = score
        print(f"  {task_name.upper()}: {score:.4f}")
    
    all_results[method_name]['average'] = np.mean([all_results[method_name][t] for t in CONFIG['tasks']])
    print(f"  Average: {all_results[method_name]['average']:.4f}")
    
    del merged
    clear_memory()

---
## 📈 Results Analysis

In [ ]:
# Summary table
print("\n" + "="*90)
print("RESULTS SUMMARY: UMTAM vs Fisher-Weighted Merging")
print("="*90)

tasks = list(CONFIG['tasks'].keys())
methods = list(all_results.keys())

# Header
header = f"{'Method':<15}" + "".join([f"{t.upper():>10}" for t in tasks]) + f"{'Average':>12}" + f"{'Δ vs UMTAM':>12}"
print(header)
print("-"*90)

# Expert row
expert_row = f"{'Expert (UB)':<15}" + "".join([f"{individual_results[t]:>10.4f}" for t in tasks])
expert_avg = np.mean(list(individual_results.values()))
expert_row += f"{expert_avg:>12.4f}" + f"{'---':>12}"
print(expert_row)
print("-"*90)

umtam_avg = all_results['UMTAM']['average']

for method in methods:
    row = f"{method:<15}"
    for t in tasks:
        row += f"{all_results[method][t]:>10.4f}"
    avg = all_results[method]['average']
    diff = avg - umtam_avg
    diff_str = f"{diff:>+.4f}" if method != 'UMTAM' else '---'
    row += f"{avg:>12.4f}" + f"{diff_str:>12}"
    
    if method == 'UMTAM':
        print("\033[1m" + row + "\033[0m")
    else:
        print(row)

print("="*90)

In [ ]:
# Key findings
fisher_methods = [m for m in methods if m.startswith('Fisher')]
fisher_avgs = {m: all_results[m]['average'] for m in fisher_methods}
best_fisher = max(fisher_avgs, key=fisher_avgs.get)
best_fisher_avg = fisher_avgs[best_fisher]

print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)

print(f"\nUMTAM average:              {umtam_avg:.4f}")
print(f"Best Fisher ({best_fisher}):  {best_fisher_avg:.4f}")
print(f"OTA-style average:          {all_results['OTA-style']['average']:.4f}")
print(f"TIES (baseline):            {all_results['TIES']['average']:.4f}")

diff_vs_best_fisher = umtam_avg - best_fisher_avg
diff_vs_ota = umtam_avg - all_results['OTA-style']['average']

print(f"\nUMTAM vs Best Fisher:  {diff_vs_best_fisher:>+.4f} ({diff_vs_best_fisher/best_fisher_avg*100:>+.2f}%)")
print(f"UMTAM vs OTA-style:    {diff_vs_ota:>+.4f} ({diff_vs_ota/all_results['OTA-style']['average']*100:>+.2f}%)")

print("\n" + "-"*70)
if diff_vs_best_fisher > 0.005:
    print("✓ CONCLUSION: UMTAM outperforms even the most precise Fisher estimation.")
    print("  Trajectory information provides unique value beyond curvature precision.")
elif diff_vs_best_fisher > -0.005:
    print("≈ CONCLUSION: UMTAM performs comparably to precise Fisher estimation.")
    print("  Advantage is computational efficiency (no post-hoc Fisher pass).")
else:
    print("✗ CONCLUSION: Precise Fisher estimation outperforms UMTAM.")
print("="*70)

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Bar chart comparison
ax1 = axes[0]
colors = ['#808080', '#1E90FF', '#FFA500'] + ['#228B22'] * len(fisher_methods)
x_pos = range(len(methods))
avgs = [all_results[m]['average'] for m in methods]

bars = ax1.bar(x_pos, avgs, color=colors)
bars[methods.index('UMTAM')].set_edgecolor('gold')
bars[methods.index('UMTAM')].set_linewidth(3)

ax1.set_ylabel('Average Score', fontsize=12, fontweight='bold')
ax1.set_title('Merging Method Comparison', fontsize=13, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(methods, rotation=45, ha='right')
ax1.set_ylim(min(avgs) * 0.95, max(avgs) * 1.05)

for bar, avg in zip(bars, avgs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{avg:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 2: Fisher precision curve
ax2 = axes[1]
fisher_samples = [100, 500, 1000]  # Approximate
fisher_perfs = [all_results[f'Fisher-{s}']['average'] if f'Fisher-{s}' in all_results else all_results['Fisher-Train']['average'] for s in fisher_samples]

ax2.plot(fisher_samples[:len([m for m in fisher_methods if 'Train' not in m])], 
         fisher_perfs[:len([m for m in fisher_methods if 'Train' not in m])], 
         'go-', linewidth=2, markersize=10, label='Fisher (validation)')
ax2.scatter([CONFIG['fisher_train_samples']], [all_results['Fisher-Train']['average']], 
            c='darkgreen', s=150, marker='s', label='Fisher (train)', zorder=5)

ax2.axhline(y=umtam_avg, color='blue', linestyle='--', linewidth=2, label=f'UMTAM ({umtam_avg:.3f})')
ax2.axhline(y=all_results['OTA-style']['average'], color='orange', linestyle=':', linewidth=2, 
            label=f'OTA-style ({all_results["OTA-style"]["average"]:.3f})')

ax2.set_xlabel('Fisher Samples', fontsize=12, fontweight='bold')
ax2.set_ylabel('Average Score', fontsize=12, fontweight='bold')
ax2.set_title('Fisher Precision vs Performance', fontsize=13, fontweight='bold')
ax2.set_xscale('log')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('umtam_vs_fisher_t4.png', dpi=150, bbox_inches='tight')
print("✓ Saved: umtam_vs_fisher_t4.png")
plt.show()

In [ ]:
# Save results
results_to_save = {
    'config': {k: v for k, v in CONFIG.items() if k != 'tasks'},
    'tasks': list(CONFIG['tasks'].keys()),
    'individual_experts': individual_results,
    'merged_results': all_results,
    'training_times_sec': training_times,
    'fisher_times_sec': fisher_times,
    'analysis': {
        'umtam_avg': umtam_avg,
        'best_fisher_method': best_fisher,
        'best_fisher_avg': best_fisher_avg,
        'umtam_vs_best_fisher': diff_vs_best_fisher,
        'umtam_vs_ota': diff_vs_ota,
    }
}

with open('umtam_vs_fisher_results.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)
print("✓ Saved: umtam_vs_fisher_results.json")

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT COMPLETE!")
print("="*70)
print(f"\nTotal runtime: {(sum(training_times.values()) + sum(fisher_times.values()))/60:.1f} min")
print("\nOutput files:")
print("  - umtam_vs_fisher_t4.png")
print("  - umtam_vs_fisher_results.json")